## Citation Descriptives
Before any modelling, we describe what the citation and impact outcomes look like by sharing
status: the group-level breakdown of raw citation counts, group h-index, and FWCI, and then a
direct univariate comparison of raw citations between sharing and non-sharing articles.

## Setup
The analytic sample is rebuilt (or loaded from the parquet cache) by `helpers.dataset.load_or_build()`, so this notebook runs standalone from a cold kernel.

In [ ]:
import pandas as pd
from scipy import stats

from helpers import dataset
from helpers.stats import compare_continuous

In [ ]:
combined, FEATURES_DF = dataset.load_or_build()

## Impact Scores by Sharing Category
A descriptive summary of the three impact measures we track - raw citation counts, the
group-level h-index, and FWCI - broken down by sharing class and for the pooled sharing
group.

In [ ]:
def calculate_h_index(citations: pd.Series) -> int:
    """Calculates h-index from a Series of citation counts."""
    if citations.empty:
        return 0
    # Sort descending (highest citations first)
    sorted_cits = sorted(citations.dropna().astype(int), reverse=True)
    # Count how many papers have citations >= their rank
    return sum(c >= i + 1 for i, c in enumerate(sorted_cits))

In [ ]:
_IMPACT_AGGS = {
    "TotalCitations": ["count", "median", "mean", "std", calculate_h_index],
    "FieldWeightedCitationIndex": ["count", "median", "mean", "std"],
}

metrics_per_sharing_class = (
    combined[["data_sharing_class", "TotalCitations", "FieldWeightedCitationIndex"]]
    .groupby("data_sharing_class")
    .agg(_IMPACT_AGGS)
)
metrics_for_any_data_sharing = (
    combined.loc[combined["is_sharing_data"], ["TotalCitations", "FieldWeightedCitationIndex"]]
    .agg(_IMPACT_AGGS)
)
display(metrics_per_sharing_class)
display(metrics_for_any_data_sharing)

In [ ]:
CITATIONS_DF = dataset.build_citations_frame(combined, FEATURES_DF)

### Univariate Comparison

In [ ]:
# compare summary statistics of citation counts between sharing and non-sharing articles
CITATIONS_DF.groupby("is_sharing_data")["log_citations"].describe()

In [ ]:
# check for normality
shapiro = stats.shapiro(CITATIONS_DF[CITATIONS_DF["is_sharing_data"] == 1]["log_citations"])
print(f"Sharing Group is Normal:\t{'YES' if shapiro.pvalue > 0.05 else 'NO'} (W={shapiro.statistic :.3f}; p={shapiro.pvalue :.3f})")

shapiro = stats.shapiro(CITATIONS_DF[CITATIONS_DF["is_sharing_data"] == 0]["log_citations"])
print(f"Non-Sharing Group is Normal:\t{'YES' if shapiro.pvalue > 0.05 else 'NO'} (W={shapiro.statistic :.3f}; p={shapiro.pvalue :.3f})")

In [ ]:
citations_univariate_test_results, citations_univariate_summary = compare_continuous(
    data=CITATIONS_DF, share_feature="is_sharing_data", tested_feature="log_citations", alternative="greater", verbose=False
)
citations_univariate_test_results = pd.Series(citations_univariate_test_results).rename("log_citations")
citations_univariate_test_results